# Tarea 12 · Análisis categórico de la baja de clientes

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/julianschmidt1/tpi-analisis-de-datos/blob/main/notebooks/trabajo/12_analisis_categorico.ipynb)

**TPI · Introducción al Análisis de Datos · UTN FRA · División 343**  
**Integrantes:** Matías, Julián y Emmanuel. **Responsable de esta tarea:** Emmanuel.

Comparamos la composición de la cartera y la tasa de baja (`Churn`) por características sociodemográficas, servicio y contratación.

## Cómo ejecutarlo

- **Local:** instalá las dependencias de `requirements.txt`, abrí el notebook desde el repo y ejecutá todas las celdas.
- **Colab:** ejecutá todas las celdas. El notebook lee los datos desde GitHub. El botón apunta a `main` y estará disponible cuando se integre la tarea. Para probarla antes, subí el archivo `.ipynb` a Colab.

## 1. Carga del dataset

Usamos `telco_clean.csv` si está disponible y `telco_provisorio.csv` durante el trabajo inicial. Seleccionamos las variables sociodemográficas y de contratación junto con `Churn`, que indica si el cliente se dio de baja (`Yes`) o siguió activo (`No`).

In [ ]:
from pathlib import Path
from urllib.error import HTTPError

import pandas as pd

try:
    import google.colab
except ImportError:
    EN_COLAB = False
else:
    EN_COLAB = True

sociodemograficas = ["gender", "SeniorCitizen", "Partner", "Dependents"]
contratacion = ["Contract", "InternetService", "PaymentMethod", "PaperlessBilling"]
variables = sociodemograficas + contratacion + ["Churn"]
ruta_limpia = Path("data/processed/telco_clean.csv")
ruta_provisoria = Path("data/processed/telco_provisorio.csv")

if EN_COLAB:
    url_repo = "https://raw.githubusercontent.com/julianschmidt1/tpi-analisis-de-datos/main/"
    fuente = url_repo + ruta_limpia.as_posix()
    try:
        datos = pd.read_csv(fuente, usecols=variables)
    except HTTPError as error:
        if error.code != 404:
            raise
        fuente = url_repo + ruta_provisoria.as_posix()
        datos = pd.read_csv(fuente, usecols=variables)
else:
    actual = Path.cwd().resolve()
    raiz = next(
        (ruta for ruta in (actual, *actual.parents)
         if (ruta / ruta_limpia).is_file() or (ruta / ruta_provisoria).is_file()),
        None,
    )
    if raiz is None:
        raise FileNotFoundError("Falta el CSV de la tarea 3 o el dataset limpio de la tarea 9.")
    fuente = raiz / ruta_limpia
    if not fuente.is_file():
        fuente = raiz / ruta_provisoria
    datos = pd.read_csv(fuente, usecols=variables)

datos = datos[variables]
print(f"Fuente: {fuente}")

## 2. Frecuencias y tasas de baja por grupo

La frecuencia absoluta cuenta clientes por categoría. La frecuencia relativa divide ese conteo por el total de la cartera. Con `crosstab(normalize="index")` dividimos por el total de cada grupo. Así, el porcentaje de baja responde, por ejemplo, cuántos de cada 100 clientes con contrato mensual se dieron de baja. Incluimos el conteo para ver el tamaño de cada grupo antes de comparar porcentajes.

In [ ]:
assert datos["Churn"].isin(["Yes", "No"]).all(), "Churn debe contener Yes o No."

tablas = {}
for variable in sociodemograficas + contratacion:
    grupos = datos[variable].astype("string").fillna("Sin dato")
    conteos = pd.crosstab(grupos, datos["Churn"]).reindex(columns=["No", "Yes"], fill_value=0)
    porcentajes = pd.crosstab(grupos, datos["Churn"], normalize="index").reindex(
        columns=["No", "Yes"], fill_value=0
    ) * 100
    clientes = conteos.sum(axis=1)
    tablas[variable] = pd.DataFrame({
        "Clientes": clientes,
        "% de la cartera": 100 * clientes / len(datos),
        "Bajas": conteos["Yes"],
        "% sigue activo": porcentajes["No"],
        "% se dio de baja": porcentajes["Yes"],
    })

for nombre, columnas in [("Sociodemográficas", sociodemograficas), ("Contratación y servicio", contratacion)]:
    print(nombre)
    print(pd.concat({columna: tablas[columna] for columna in columnas}).round(2).to_string())
    print()

## 3. Hallazgos con el CSV provisorio

1. **Contrato.** Se dio de baja el 42,71 % de los clientes con contrato mensual, frente al 11,27 % de los clientes con contrato anual y el 2,83 % de quienes tenían contrato de dos años. El grupo mensual concentra la mayor tasa de baja entre los tipos de contrato.
2. **Adultos mayores.** La tasa fue del 41,68 % para `SeniorCitizen = 1` frente al 23,61 % para `SeniorCitizen = 0`. La diferencia sirve para identificar un grupo a estudiar con más detalle y cruzarlo con el tipo de contrato.
3. **Servicio de internet.** Los clientes con fibra óptica tuvieron una tasa de baja del 41,89 %, frente al 18,96 % en DSL y al 7,40 % entre quienes no tenían internet contratado. La comparación permite identificar el segmento de servicio con mayor tasa observada.

Estas comparaciones describen asociaciones entre categorías y bajas. Para el análisis final, revisar las cifras después de ejecutar el notebook con `telco_clean.csv`.